Q1 — Swap without a third variable
a, b = b, a

In [ ]:
# Write code to swap two variables' values without using a third variable.
a = 10
b = 5
a,b = b,a
print(a,b)

5 10


Correct, and this is the idiomatic Python way — good instinct.
Internally: Python builds the right-hand side as a temporary tuple (b, a) first, fully evaluating both values
before any assignment happens, then unpacks that tuple into a, b on the left. So there's no real "magic swap"
— Python is quietly using a tuple as scratch space instead of you declaring one explicitly. This is worth
remembering because it generalizes: a, b, c = c, a, b works the same way for a 3-way rotation.


In [ ]:
# Take three variables a, b, c and print their types using type().
a = 10
b = 'String'
c = True
print(type(a))
print(type(b))
print(type(c))

<class 'int'>
<class 'str'>
<class 'bool'>


Q2 — type() checks
Correct. One thing worth knowing for later: bool in Python is technically a subclass of int (True == 1, False == 0
under the hood). Doesn't change your answer, but it's a classic gotcha topic — I'll queue a question on it for later.

What "subclass of int" literally means

In Python, bool inherits from int:

python
print(bool.__bases__)   # (<class 'int'>,)
print(isinstance(True, int))   # True

This means every True/False value is an int under the hood — True is essentially int value 1, and False is int value 0, just with a different display representation and type label.

Where this actually bites you (or helps you)
1. Booleans work in arithmetic — silently
python
print(True + True)      # 2
print(True + 1)         # 2
print(False * 10)       # 0
print(sum([True, True, False, True]))   # 3

That last one is genuinely useful in practice — sum() on a list of booleans gives you a count of True values. You'll use this constantly once you get to NumPy/Pandas for counting matches/conditions (sum(df['age'] > 30) counts how many rows satisfy the condition).

2. Booleans can index lists (dangerous gotcha)
python
lst = ['a', 'b', 'c']
print(lst[True])    # 'b' — because True acts as index 1!
print(lst[False])   # 'a' — False acts as index 0

This is a real bug source: if a function accidentally receives a boolean where it expects an int index, it won't crash — it'll silently do the wrong thing (index 0 or 1), which is exactly the kind of bug that's hard to spot, similar to the Q10 issue you just had.

3. True == 1 and False == 0 — collision in dicts/sets
python
d = {True: "yes", 1: "no"}
print(d)   # {True: 'no'} — only ONE key exists!

Because True and 1 are considered equal and hash to the same value, using both as dict keys collides — the second assignment overwrites the first. This is a genuinely nasty bug if you're not aware of it, especially when building dictionaries from mixed data that might contain both booleans and integers.

4. type() still tells them apart
python
print(type(True))   # <class 'bool'>
print(True == 1)    # True  (value equality)
print(True is 1)    # False (different objects, bool isn't literally int)

So: bool behaves like int in arithmetic and comparisons, but reports its own distinct type, and is still correctly distinguishes them because they're separate objects.

In [ ]:
# Convert a string "123" to an integer, add 10 to it, and print the result along with its type.
a = '123'
result = int(a) + 10
print(result)
print(type(result))

133
<class 'int'>


Q3 — String to int conversion
Correct. Internally, int("123") doesn't "reinterpret bytes" — it parses the string character by character and builds
a new int object numerically. If the string had invalid characters ("123abc"), this would throw ValueError, not
silently fail — worth testing that edge case yourself as a bonus rep.

In [ ]:
# Given x = 10, y = "10", write code that checks if x == y and explain why the result is what it is.
x = 10
y = '10'
print(x==y)
# return False because one is belong to type int while other belongs to string

False


Q4 — x == y where x=10, y="10"
Correct output, but let's sharpen the explanation. Your comment says "one belongs to int while other belongs to string"
 — true, but the more precise mechanism: Python's == calls x.__eq__(y) first. Since int.__eq__ doesn't know how to compare
  itself to a str, it returns NotImplemented, so Python then tries y.__eq__(x) (str.__eq__), which also doesn't know how
  to compare to int and also returns NotImplemented. When both sides give up, Python falls back to False rather than raising
 an error. That's genuinely a good interview-level detail to know — most people just say "different types" without knowing
what actually happens under the hood.

In [ ]:
# Create one variable of each core type (int, float, str, bool, complex) and print all their id()s.
a = 10
b = 10.25
c = 'Sadu'
d = True
e = 2 + 4j
print(id(a))
print(id(b))
print(id(c))
print(id(d))
print(id(e))

11645640
132483969205488
132483969287872
10549600
132483969209552


Q5 — id() of each type
Correct and complete. Notice something interesting in your own output: id(a) for a = 10 is a short number (11645640) while the
 others are long numbers (132483969...). That's not random — small integers (-5 to 256) are pre-cached by CPython at startup,
 so they live in a fixed, low memory region, which is why the id looks different in shape from the others. Complex numbers, by
 contrast, are never interned — every 2 + 4j you create is a fresh object with a fresh id, even if the value is identical elsewhere
 in your code. Good real-world segue into your Q14 (interning range) from the harder tier.

In [ ]:
# Predict the output:
# python
#    a = 5
#    b = a
#    b = b + 1
#    print(a, b)

# Then explain why, referencing whether int is mutable or immutable.
a = 5
b = a
b = b + 1
print(a, b)
# the output is 5 6

5 6


Q6 — a=5; b=a; b=b+1
Correct output, correct concept. Internally: b = a makes b point to the same int object as a (id(a) == id(b) at that instant). Then
 b = b + 1 doesn't mutate that object — ints are immutable, so Python computes a brand new int object 6, and rebinds the name b to
 point at it. a never moves. This is exactly what your comment says, just worth having the "rebind, not mutate" phrasing locked in —
 it's the core mental model for the rest of Python.


What I said wrong

I said "Python computes a brand new int object 6." That's not quite accurate here specifically because 6 falls inside the interned range (-5 to 256). It's not created at that moment — it already exists, cached since interpreter startup. So b = b + 1 doesn't allocate anything new; it just looks up the existing pre-cached object for 6 and rebinds b to point at it.

What stays true regardless

Here's the important part: the "rebind, not mutate" conclusion doesn't depend on interning at all. Whether Python:

pulls a pre-cached object for 6 (because it's in the interned range), or
freshly allocates a new object for 6 (if it were, say, 1000)

...in both cases, b ends up pointing to a different object than a, and a itself is never touched. Interning is purely an optimization about where the new-to-b object comes from — cache vs. fresh allocation — it has zero effect on the actual mechanism of reassignment (rebind the name, don't mutate the old object).

Prove it to yourself
python
a = 5
b = a
print(id(a), id(b))   # same id — both point to the cached int object for 5

b = b + 1
print(id(a), id(b))   # different ids — a still points to 5's object,
                       # b now points to 6's cached object (pre-existing, not newly made)

id(a) never changes across both prints. id(b) changes — but the object it changes to was already sitting in Python's interned cache, not manufactured on the spot.

The corrected mental model
Immutability guarantees an int can never be mutated in place — b = b + 1 must produce a different object, always, no exceptions.
Interning is a separate, independent optimization that only affects whether that different object is freshly allocated or reused from cache — and only for small ints (-5 to 256).
a=any_value and b=a this means id(a) = id(b) will always be true beacause b will not be copy of a it is pointing to the same object as a.

In [ ]:
# Write code that takes a float like 3.14159 and rounds it to 2 decimal places two different ways (built-in function vs. string formatting).
a = 3.14159
print("{:.2f}".format(a))
print(round(a,2))

3.14
3.14


Q7 — Rounding
Correct and complete. One gotcha to file away for later (not a bug here, just good to know): round() uses "banker's rounding" —
round(2.5) gives 2, not 3, because Python rounds to the nearest even number on exact .5 ties. Doesn't affect your answer, but it's
a classic gotcha question in its own right.


In [ ]:
# Given num = "3.14abc", write code that safely attempts to convert it to float and handles the failure gracefully (you'll need a bit of exception handling here — fine to preview it).
num = '3.14abc'
try :
  num_float = float(num)
  print(num_float)
except ValueError:
  print('Sorry, float conversion is not possible')

Sorry, float conversion is not possible


Q8 — Safe float conversion
Correct, clean try/except. Good instinct catching ValueError specifically rather than a bare except.

In [ ]:
# Predict the output and explain:

# python
#    x = 300
#    y = 300
#    print(x is y)
#    print(x == y)
x = 300
y = 300
print(x is y)
print(x == y)
# False
# True
# because is operator is comparing memory loaction whereas == operator is comparing values

False
True


Q9 — x=300, y=300
Correct, and your explanation nails the mechanism precisely.

In [ ]:
# Write a function that takes any variable and returns whether it's mutable or immutable, by checking its type against a known list.
def fun(x):
  mutable_types = (list, dict, set, bytearray)
  immutable_types = (int, float, str, tuple, bool, frozenset, bytes)
  if type(x) in mutable_types:
    return 'Mutable'
  elif type(x) in immutable_types:
    return 'Immutable'


Q10 — Correct

In [ ]:
# Predict the output and explain in your own words, referencing id():
# python
#     a = 10
#     b = 10
#     c = 1000
#     d = 1000
#     print(a is b)
#     print(c is d)
a = 10
b = 10
c = 1000
d = 1000
print(a is b)
print(c is d)
# True
# False
# because there the concept in python called 'Interning'
# in interger having range -5 to 256, python have particular specific memory for it even
# if user creates multiple identifiers pointing to the same value in this range it will
# not create different object, will point to the same object

True
False


Q11 — a=10,b=10,c=1000,d=1000
Correct output and explanation.

In [ ]:
# Write code to demonstrate that Python variables are "labels/references to objects" rather than "boxes holding values" — use id() to prove a reassignment creates a new binding, not a mutation, for an integer.
a = 1000
print(id(a))
a = a + 1
print(id(a))

132483959676752
132483959682736


Q12 — Correct


In [ ]:
# Given a list original = [1, 2, 3], do copy = original, then copy.append(4). Predict original's value and explain why this behaves differently from the integer case in Q11/Q12.
original = [1, 2, 3]
copy = original
copy.append(4)
print(original)
print(copy)

[1, 2, 3, 4]
[1, 2, 3, 4]


Q13 — List aliasing

Correct output, correctly demonstrates that copy = original doesn't create a new list — it just gives two names pointing at the same
list object, so .append() mutates the one shared object and both names see the change. This is the key contrast with Q6/Q12: lists are
mutable, so an in-place operation like .append() changes the object itself rather than creating a new one. Your code is right — just add
that sentence as a comment, since the question explicitly asked you to explain, not just show.


In [ ]:
# Write a program that demonstrates integer interning: create integers from -6 to 260 in a loop, check is equality with a separately created same-value integer, and print the exact range where is starts returning False. (Your notes mention -5 to 256 — verify it yourself.)
for i in range(-6,260):
  a = i
  b = i
  if (a is b):
    print(i)

-6
-5
-4
-3
-2
-1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259


Q14 — Verifying the interning range — 🐛 the most important bug in this notebook
python
for i in range(-6, 260):
  a = i
  b = i
  if (a is b):
    print(i)

Every single number printed True — even -6 and 259, which are outside the real interning range. Something's wrong, and figuring out
why is a genuinely valuable exercise.

What's actually happening: a = i and b = i don't create two separate integer objects with the same value — they both just bind to whatever
object the loop variable i already is at that point in the loop. So of course a is b — they're not independently-created integers, they're two
names pointing at the same existing object (i itself). Your test methodology accidentally tests "can two names point to the same object" (trivially
always true) instead of "does Python give two independently-created same-value integers the same object" (the real interning question).

Fixed version — force genuinely independent object creation using a string round-trip (this defeats compiler-level object reuse):

python
for i in range(-6, 260):
    a = int(str(i))
    b = int(str(i))
    if not (a is b):
        print(f"First non-interned value: {i}")
        break

Run this and you should see it break right around 257 (i.e., -5 to 256 are interned, 257 is the first one that isn't). This is a great one to re-run
yourself and confirm — it'll cement the concept far better than my telling you the answer.


In [ ]:
for i in range(-6, 260):
    a = int(str(i))
    b = int(str(i))
    if not (a is b):
        print(f"non-interned value: {i}")


non-interned value: -6
non-interned value: 257
non-interned value: 258
non-interned value: 259


In [ ]:
# Predict the output:
# python
#     a = 5
#     def foo():
#         print(a)
#         a = 10
#     foo()
a = 5
def foo():
    print(a)
    a = 10
foo()

UnboundLocalError: cannot access local variable 'a' where it is not associated with a value

Q15 — UnboundLocalError

Correct — you got the exact right error. Your comment cuts off mid-sentence ("a is gloabal variable here and in python we can access...") — finish that
thought: Python decides, at compile time, whether a name is local or global for an entire function
body — based on whether that name is assigned anywhere in the function. Since a = 10 appears in
foo(), Python treats a as local for the whole function, including the print(a) line before the
assignment — so it doesn't fall back to the global a = 5, it just errors. This trips up even
experienced developers coming from other languages.


Step 1: What you'd expect to happen
python
a = 5
def foo():
    print(a)
    a = 10
foo()

Your intuition probably says: "print(a) runs first, and at that point a is still 5 (the global one), so it should print 5." That's a completely reasonable guess — and it's wrong, and the reason why is the important part.

Step 2: The key idea — Python decides local vs. global BEFORE running anything

Before your function ever runs, Python scans the entire body of foo() and asks one question: "Is the name a assigned anywhere in this function?"

It finds a = 10 somewhere inside foo(). That's enough. The moment Python sees any assignment to a inside the function, it makes a decision for the whole function, top to bottom:

"a is a local variable in this function. Full stop. It does not matter where the assignment appears in the code."

This decision happens at compile time — before foo() is even called, before print(a) executes, before anything. It's not a line-by-line, top-to-bottom process where Python "figures out a is global until it hits the a = 10 line." It decides upfront, once, for the entire function.

Step 3: So what actually happens when you call foo()
Python has already decided: inside foo(), a = local variable.
foo() starts running.
It hits print(a).
It looks for the local a (because that's what it already decided a means in this function).
But the local a doesn't have a value yet — the line that would give it one (a = 10) hasn't executed yet.
Result: UnboundLocalError — "this local variable exists, but nothing's been put in it yet."

In [ ]:
# Predict the output:

# python
#     a, b = b, a = 1, 2
a, b = b, a = 1, 2
print(a,b)
# yes valid it will swap the values of a and b

2 1


Q16 — Chained assignment

Correct output (2 1), but your explanation ("it will swap the values") describes the effect, not the mechanism — and the mechanism is worth
knowing precisely. a, b = b, a = 1, 2 is a chained assignment: the right-most expression 1, 2 is evaluated once, then assigned to each target
left-to-right. So: first assign (1,2) to (a, b) → a=1, b=2. Then assign that same (1,2) tuple to (b, a) → b=1, a=2. Final state: a=2, b=1. It
looks like a swap, but it's really "the same tuple gets assigned to two different target patterns in sequence" — a subtle distinction that matters
if the targets weren't a clean permutation of each other.

In [ ]:
# Explain why this returns True even though 0.1 + 0.2 "should" equal 0.3:

# python
#     print(0.1 + 0.2 == 0.3)
print(0.1 + 0.2 == 0.3)  # False actually — predict correctly!
# because this will result into something like 0.3000000004 because of floating data representing
print(round(0.1+0.2,1))

False
0.3


Q17 — Float precision

Correct, and good bonus check with round().


In [ ]:
# Predict the output:

# python
#     a = "hello"
#     b = "hello"
#     print(a is b)

#     c = "hello world"
#     d = "hello world"
#     print(c is d)
a = "hello"
b = "hello"
print(a is b)

c = "hello world"
d = "hello world"
print(c is d)
#True
# False because string interning in python allows only for small strings

True
False


Q18 — String interning

Correct output and correct high-level explanation. To sharpen it: CPython auto-interns strings that look like identifiers (letters/digits/underscores,
no spaces) — because these are common as variable/attribute/dict-key names internally, so caching them saves memory. Strings with spaces ("hello world")
aren't automatically interned because they're far less likely to be reused as identifiers, so Python doesn't bother — hence c is d is False.


Overall verdict

16/18 conceptually solid, 2 real bugs — and importantly, both bugs were the kind that don't crash, they just silently do the wrong thing (wrong variable in Q10,
flawed test methodology in Q14). That's a more valuable catch than any correct-answer question would've been — this is exactly the debugging instinct that separates
"can write code" from "can be trusted with code," and it's worth deliberately building. Fix Q10 and Q14, re-run them, confirm your output matches the corrected logic,
then commit all 18.

In [ ]:
# Predict the output — is there an error, and if so, on which line?
# python
# count = 0
# def increment():
#     count += 1
#     print(count)
# increment()
count = 0
def increment():
    count += 1
    print(count)
increment()

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value

What actually happens
python
count = 0
def increment():
    count += 1
    print(count)
increment()

Output: UnboundLocalError: cannot access local variable 'count' where it is not associated with a value

Why — the precise mechanism

count += 1 is just shorthand for count = count + 1. That means there is an assignment to count inside increment() — even though it's disguised as a compound operator, Python still sees an assignment target on the left of =.

So exactly like your Q15 case: Python scans the function body, sees count gets assigned somewhere inside it, and decides — before running anything — that count is local to increment() for the entire function body.

Then when increment() actually runs:

It hits count += 1, which really means count = count + 1.
To compute the right-hand side (count + 1), Python needs to first read the current value of count.
It looks for the local count (already decided in step 1).
The local count has no value yet — nothing has run yet to give it one.
UnboundLocalError.

In [ ]:
# Predict the output for both function calls:
# python
x = 100
def outer():
    print(x)
    def inner():
        print(x)
        x = 5
    inner()
outer()

100


UnboundLocalError: cannot access local variable 'x' where it is not associated with a value

Trace it scope by scope

outer()'s own body: Does outer() assign to x anywhere directly in its own body? Look carefully — x = 5 is inside inner(), not inside outer() itself. outer() only has print(x) and a call to inner(). So as far as outer()'s own scope is concerned, x is never assigned in it — which means Python treats x in outer()'s print(x) as referring to the global x.

So the first line that runs: print(x) inside outer() → prints 100. No error here.

inner()'s own body: Now look at inner() in isolation. It has print(x) followed by x = 5. That assignment makes x local to inner() — exactly like your Q1 and Q15 cases. So print(x) inside inner() tries to read the local x before it's been assigned → UnboundLocalError.

Full actual output
100
Traceback (most recent call last):
  ...
UnboundLocalError: cannot access local variable 'x' where it is not associated with a value
Why this one matters more than it looks

This is the key lesson: the "is this name local?" decision is made separately, per function, based only on assignments in that function's own body. A nested function doesn't inherit "local-ness" from its parent, and a parent doesn't get contaminated by what a child function does internally. outer() and inner() each get their own independent scope analysis, even though inner() is defined inside outer().

In [ ]:
# Fix this function so it prints 6 without using the global keyword at all:
# python
a = 5
def foo():
    print(a)
    a = a + 1
foo()

UnboundLocalError: cannot access local variable 'a' where it is not associated with a value

Fix this function so it prints 6 without using the global keyword at all:
python
a = 5
def foo():
    print(a)
    a = a + 1
foo()

In [ ]:
a = 5
def foo():
    b = a + 1   # 'a' is never assigned inside foo(), so this reads the global a
    print(b)
foo()

6


In [53]:
def foo():
    a = 1
    def bar():
        a = a + 1
        return a
    return bar()
print(foo())

UnboundLocalError: cannot access local variable 'a' where it is not associated with a value

Why

bar() has its own body: a = a + 1, followed by return a. That single line a = a + 1 is enough — Python scans bar()'s body, sees a gets assigned there, and locks a in as local to bar(), for the entire function. It doesn't matter that foo() (the enclosing function) already has its own a = 1 sitting right there one level up — nested functions don't automatically "see through" to the parent's local variables the way they can see global ones.

So when bar() runs a = a + 1, it tries to read the local a on the right-hand side before it's ever been given a value → UnboundLocalError

In [49]:
a = [1, 2, 3]
b = a
print(id(a) == id(b))   # True

a = a + [4]
print(a)   # [1, 2, 3, 4]
print(b)   # [1, 2, 3]  ← unchanged!
print(id(a) == id(b))   # False — a now points to a new object, b still points to the old one

True
[1, 2, 3, 4]
[1, 2, 3]
False


In [50]:
a = [1, 2, 3]
b = a
print(id(a) == id(b))   # True

a += [4]
print(a)   # [1, 2, 3, 4]
print(b)   # [1, 2, 3, 4]  ← b changed too!
print(id(a) == id(b))   # True — still the SAME object

True
[1, 2, 3, 4]
[1, 2, 3, 4]
True


a = a + [4] and a += [4] works differently

In [51]:
a = (1, 2, 3)
b = a
print(id(a) == id(b))   # True — same object, as always right after b = a

b += (4,)
# no __iadd__ exists for tuples → falls back to b = b + (4,)
# → creates a NEW tuple (1,2,3,4), rebinds b to it, a is untouched

print(a)                # (1, 2, 3)       — unchanged
print(b)                # (1, 2, 3, 4)    — new object
print(id(a) == id(b))   # False

True
(1, 2, 3)
(1, 2, 3, 4)
False


In [54]:
d = {}
d[1] = "int one"
# d is now {1: "int one"}

d[True] = "bool true"
# Python checks: does a key equal to `True` already exist?
# hash(True) == hash(1), and True == 1 → yes, key 1 already exists
# → overwrites the value at that key, doesn't create a new entry

print(d)         # {1: 'bool true'}
print(len(d))    # 1

{1: 'bool true'}
1


In [55]:
scores = [45, 78, 92, 34, 88, 55]
passing = sum(score >= 50 for score in scores)
print(passing)

4


First, score >= 50 is evaluated for every score in the list — this produces a sequence of booleans, one per score:

score	score >= 50
45	False
78	True
92	True
34	False
88	True
55	True

That's [False, True, True, False, True, True].

Now, sum() on a sequence of booleans — exactly like the sum([True, True, False, True]) example from your very first bool-as-int explanation — treats each True as 1 and each False as 0, and adds them up.

Count the Trues: 78, 92, 88, 55 → that's 4 trues.

Correct output: 4

What this pattern actually is

This whole expression is a compact, idiomatic way to count how many items satisfy a condition — sum(condition for item in iterable) is a very common pattern once you get to real data work. It's directly connecting back to the very first thing I told you bool-as-int would be useful for: this exact pattern is what sum(df['age'] > 30) does in Pandas to count matching rows.

In [56]:
def double_all(numbers):
    result = numbers
    for i in range(len(result)):
        result[i] = result[i] * 2
    return result

original = [1, 2, 3]
doubled = double_all(original)
print(original)
print(doubled)

[2, 4, 6]
[2, 4, 6]


result = numbers doesn't create a copy — same rule as every earlier question. result and numbers (which is original, from the caller's side) end up pointing at the same list object. So when the loop does result[i] = result[i] * 2, it's mutating that one shared list in place — and since original was never a separate object to begin with, it gets silently doubled too.

In [57]:
class Thing:
    def __eq__(self, other):
        return True

t = Thing()
print(t == None)   # True  ✅ your answer
print(t is None)   # False ❌ you said True

True
False


t == None → True — correct, and here's exactly why it's dangerous, not just "true because we overrode it": t == None calls t.__eq__(None). Since Thing overrides __eq__ to always return True no matter what it's compared against, Python believes t equals None — even though t is obviously a real Thing object, nothing like None at all. Any object with a badly (or maliciously) written __eq__ can lie to you this way.

t is None → False, not True. is never calls __eq__ at all — it's a pure identity check, asking "are these literally the same object in memory?" t is a Thing instance; None is a completely separate, singleton object. They are never the same object, full stop, regardless of what any class defines. This is exactly why is None is the safe, idiomatic way to check for None — it's immune to the __eq__-lying trick above, while == None is not.

The one-line takeaway: ==  can be fooled by custom classes overriding __eq__; is cannot be fooled by anything, because identity isn't something a class can redefine.

In [52]:
class Animal:
    pass

class Dog(Animal):
    pass

d = Dog()

print(type(d) == Dog)
print(type(d) == Animal)
print(isinstance(d, Dog))
print(isinstance(d, Animal))

True
False
True
True


In [58]:
a, *b, c = [1, 2, 3, 4, 5]
print(a)
print(b)
print(c)

x, *y = [10]
print(x)
print(y)

*p, q = []
print(p)
print(q)

1
[2, 3, 4]
5
10
[]


ValueError: not enough values to unpack (expected at least 1, got 0)

In [59]:
print(2 ** 1000 > 0)

x = float('nan')
print(x == x)

y = float('inf')
print(y > 10**100)

True
False
True


2 ** 1000 > 0 → True ✅

Correct, and here's what's notable: in most languages (C, Java, etc.), an int this large would overflow — wrap around, become negative, or error, because those languages store ints in a fixed number of bits. Python ints have arbitrary precision — they automatically grow to however many digits are needed, with no overflow, ever. 2 ** 1000 is a genuine ~302-digit number, computed exactly, no rounding, no wraparound. This is a real practical advantage Python has over lower-level languages.

float('nan') == float('nan') → False, not True ❌

This is the single most famous "gotcha" in floating-point across every language that follows the IEEE 754 standard, not just Python:

python
x = float('nan')
print(x == x)   # False

nan stands for "Not a Number" — it represents the result of an undefined or unrepresentable computation (like 0/0 or inf - inf). By definition in the IEEE 754 standard, nan is specified to never equal anything, including itself. This isn't a Python quirk — it's a deliberate mathematical convention baked into how virtually every programming language handles floating point.

Why this actually matters in practice: it means you can never reliably check for nan using ==. If you write if value == float('nan'), that condition is always False, even when value genuinely is nan — a silent, hard-to-spot bug. The correct way to check:

python
import math
print(math.isnan(x))   # True — the actual correct way to test for nan

You'll run into this constantly once you touch Pandas — missing/invalid data is frequently represented as nan, and df[df['col'] == nan] is a classic beginner mistake that silently returns nothing, ever.

float('inf') > 10**100 → True ✅

Correct. inf represents mathematical infinity — by definition, it's greater than any finite number, no matter how enormous. 10**100 is a genuinely massive number (a googol), but inf still exceeds it, because that's precisely what inf means.

Summary of the round
nan breaks the normal rule that "a thing always equals itself" — the one genuine exception to == reflexivity in the entire language.
Correct test for nan is always math.isnan(x), never ==.

5/6 correct this round with the nan correction now locked in.